# Nados AI v1.1 — خادم الاستدلال السحابي (إصدار البناء الثابت)

شغّل الخلايا بالترتيب (Runtime → Run all). أي خلية فشلت أعد تشغيلها.

**مهم**: إذا أعاد كولاب تشغيل الجلسة، أعد تشغيل كل الخلايا (الملفات تُمسح).


In [ ]:
#@title 1) تثبيت cloudflared
import subprocess, os

def run(cmd):
    return subprocess.run(cmd, shell=True, capture_output=True, text=True)

if not os.path.exists('cloudflared'):
    r = run('wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared')
    if r.returncode != 0:
        print('WGET STDERR:', r.stderr[:300])
run('chmod +x cloudflared')
print('cloudflared:', os.path.exists('cloudflared'))
print('جاهز — انتقل للخلية 2')


In [ ]:
#@title 1.5) بناء llama-server ثابت (بلا مكتبات خارجية) ~10 دقائق
import subprocess, os

def run(cmd):
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    return result

if not os.path.exists('build/bin/llama-server'):
    if not os.path.exists('llama-src'):
        print('cloning llama.cpp...')
        r = run('git clone --depth 1 https://github.com/ggerganov/llama.cpp.git llama-src')
        if r.returncode != 0:
            print('CLONE STDERR:', r.stderr[:300])
    print('configuring (static, CPU)...')
    r = run('cmake -S llama-src -B build -DGGML_CUDA=OFF -DLLAMA_CURL=OFF -DBUILD_SHARED_LIBS=OFF')
    if r.returncode != 0:
        print('CMAKE STDERR:', r.stderr[-500:])
    print('building (يستغرق ~10 دقائق)...')
    r = run('cmake --build build --config Release -j2 --target llama-server')
    if r.returncode != 0:
        print('BUILD STDERR:', r.stderr[-800:])
else:
    print('البناء موجود مسبقاً')

exists = os.path.exists('build/bin/llama-server')
print('llama-server (static):', exists)
print('جاهز — انتقل للخلية 2' if exists else 'فشل — الصق لي المخرجات')


In [ ]:
#@title 2) تنزيل النموذج الأساسي والمحوّل المدرَّب
MODEL_SIZE = 'mini' #@param ['mini', 'full']
import os
if MODEL_SIZE == 'mini':
    if not os.path.exists('base.gguf'):
        !wget -q https://huggingface.co/bartowski/gemma-2-2b-it-GGUF/resolve/main/gemma-2-2b-it-Q4_K_M.gguf -O base.gguf
    if not os.path.exists('nados-lora.gguf'):
        !wget -q https://huggingface.co/noore7xd/nados-models/resolve/main/nados-v1-1-mini-lora.gguf -O nados-lora.gguf
else:
    if not os.path.exists('base.gguf'):
        !wget -q https://huggingface.co/bartowski/gemma-2-9b-it-GGUF/resolve/main/gemma-2-9b-it-Q4_K_M.gguf -O base.gguf
    if not os.path.exists('nados-lora.gguf'):
        !wget -q https://huggingface.co/noore7xd/nados-models/resolve/main/nados-v1-1-lora.gguf -O nados-lora.gguf
print('base:', round(os.path.getsize('base.gguf')/(1024**3), 2), 'GB | lora:', round(os.path.getsize('nados-lora.gguf')/(1024**2), 0), 'MB')
print('النماذج جاهزة — انتقل للخلية 3')


In [ ]:
#@title 3) تشغيل خادم Nados v1.1 + النفق العام
import subprocess, time, re, os

SERVER_BIN = 'build/bin/llama-server' if os.path.exists('build/bin/llama-server') else 'llama-server'
if not os.path.exists(SERVER_BIN):
    raise RuntimeError('llama-server غير موجود — شغّل الخلية 1.5 أولاً')
if not os.path.exists('base.gguf') or not os.path.exists('nados-lora.gguf'):
    raise RuntimeError('النماذج غير موجودة — شغّل الخلية 2 أولاً')

server = subprocess.Popen([SERVER_BIN, '-m', 'base.gguf', '--lora', 'nados-lora.gguf', '--host', '0.0.0.0', '--port', '8080', '--threads', '2'], stdout=open('server.log', 'w'), stderr=subprocess.STDOUT)
time.sleep(20)
health = subprocess.run(['curl', '-s', 'http://localhost:8080/health'], capture_output=True, text=True)
print('الخادم المحلي:', 'يعمل' if 'ok' in health.stdout.lower() else 'فشل: ' + health.stdout[:150])
if 'ok' not in health.stdout.lower():
    print(open('server.log').read()[-400:])

log = open('tunnel.log').read() if os.path.exists('tunnel.log') else ''
match = re.search(r'https://[a-z0-9-]+[.]trycloudflare[.]com', log)
if match:
    print('النفق يعمل بالفعل — الرابط:')
    print(match.group(0))
else:
    tunnel = subprocess.Popen(['./cloudflared', 'tunnel', '--url', 'http://localhost:8080', '--no-autoupdate'], stdout=open('tunnel.log', 'w'), stderr=subprocess.STDOUT)
    url = None
    for attempt in range(6):
        time.sleep(10)
        log = open('tunnel.log').read()
        match = re.search(r'https://[a-z0-9-]+[.]trycloudflare[.]com', log)
        if match:
            url = match.group(0)
            break
    print('الرابط العام:', url if url else 'لم يظهر — tunnel.log:')
    if not url:
        print(open('tunnel.log').read()[:600])


In [ ]:
#@title 4) اختبار النموذج المدرَّب (اختياري)
import subprocess, json
question = 'ما فوائد الاختبارات الآلية؟' #@param {type:'string'}
body = json.dumps({'messages': [{'role': 'user', 'content': question}], 'max_tokens': 300})
result = subprocess.run(['curl', '-s', '-X', 'POST', 'http://localhost:8080/v1/chat/completions', '-H', 'Content-Type: application/json', '-d', body], capture_output=True, text=True)
try:
    reply = json.loads(result.stdout)
    print('إجابة Nados v1.1:', reply['choices'][0]['message']['content'][:500])
except Exception:
    print('raw:', result.stdout[:400])
